# Disease screening model training

This notebook is designed for Google Colab. It is intentionally structured to train a cattle disease classifier using the dataset in `datasets/disease/`.

## Workflow
1. Mount Google Drive or upload the dataset.
2. Validate class folders and file readability.
3. Inspect counts and remove obvious leaks or duplicates.
4. Train a MobileNetV2 or EfficientNetB0 transfer learning model.
5. Save the trained model and export the class mapping.
6. Download the artifacts for use by the Flask backend.


In [ ]:
!pip install -q tensorflow pillow matplotlib numpy seaborn scikit-learn imagehash tqdm

from pathlib import Path
from collections import Counter
import json

DATASET_ROOT = Path('/content/datasets/disease')
print('Dataset exists:', DATASET_ROOT.exists())
for class_dir in sorted(DATASET_ROOT.iterdir()):
    if class_dir.is_dir():
        files = list(class_dir.glob('*'))
        print(class_dir.name, len(files))


## Training configuration

- Use MobileNetV2 or EfficientNetB0 with ImageNet weights.
- Keep a held-out test set untouched until final evaluation.
- Use early stopping and learning-rate scheduling.
- Save the model and mapping to a download-friendly folder.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print('TensorFlow version:', tf.__version__)


In [ ]:
# Create train/val/test split with stratification and leakage checks.
# Use dataset directory classes after verification.
from sklearn.model_selection import train_test_split

classes = sorted([p.name for p in DATASET_ROOT.iterdir() if p.is_dir()])
print('Classes:', classes)


In [ ]:
# Export final artifact bundle for the backend.
OUTPUT_DIR = Path('/content/artifacts/disease_model')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save class mapping and model after training.
# model.save(OUTPUT_DIR / 'disease_model.keras')
# (OUTPUT_DIR / 'class_mapping.json').write_text(json.dumps({'classes': classes}, indent=2), encoding='utf-8')
